Crear terminal e instalar dependencias

In [ ]:
python -m venv venv
source venv\Scripts\activate

pip install pandas matplotlib seaborn scikit-learn wordcloud

Cargar y explorar los datos

El dataset FinancialPhraseBank tiene ~4.840 frases de noticias
financieras en inglés, etiquetadas por 16 expertos en finanzas.
3 clases: positive, negative, neutral

In [8]:
import pandas as pd

df = pd.read_csv("dataset/all-data.csv",
                 encoding="latin-1",         
                 header=None,                
                 names=["sentiment", "sentence"])

print(f"Total de frases: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nPrimeras 5 filas: \n")
print(df.head())

# Distribución de classes

print(f"\nDistribución de sentimientos: \n")
print(df["sentiment"].value_counts())
print(f"\nEn porcentajes: \n")
print(df["sentiment"].value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))

# Ejemplos por clase

for sentence in ["positive", "negative", "neutral"]:
    print(f"\n{'='*50}")
    print(f"  EJEMPLOS {sentence.upper()}")
    print(f"{'='*50}")
    samples = df[df["sentiment"] == sentence]["sentence"].sample(3, random_state=42)
    for i, text in enumerate(samples, 1):
        print(f"  {i}. {text[:100]}...")

Total de frases: 4846
Columnas: ['sentiment', 'sentence']

Primeras 5 filas: 

  sentiment                                           sentence
0   neutral  According to Gran , the company has no plans t...
1   neutral  Technopolis plans to develop in stages an area...
2  negative  The international electronic industry company ...
3  positive  With the new production plant the company woul...
4  positive  According to the company 's updated strategy f...

Distribución de sentimientos: 

sentiment
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64

En porcentajes: 

sentiment
neutral     59.4%
positive    28.1%
negative    12.5%
Name: proportion, dtype: object

  EJEMPLOS POSITIVE
  1. The new agreement , which expands a long-established cooperation between the companies , involves th...
  2. ( ADP News ) - Finnish handling systems provider Cargotec Oyj ( HEL : CGCBV ) announced on Friday it...
  3. The world 's biggest magazine paper maker said the program to im

Neutral domina (~59%), seguido de positive (~28%) y negative (~13%). Esto es desbalanceo de clases — el problema más común en ML aplicado.


Las frases positivas usan palabras como "growth", "profit", "increased". Las negativas usan "loss", "fell", "declined". Esto nos adelanta que un modelo basado en palabras individuales ya debería funcionar razonablemente

Análisis de palabras y longitudes

In [9]:
import re
from collections import Counter


# Stopwords: palabras tan comunes que no aportan significado
# "the", "a", "of"... aparecen en TODAS las frases
stopwords = {
    'the', 'a', 'an', 'in', 'of', 'to', 'and', 'for', 'is', 'was',
    'on', 'it', 'its', 'has', 'with', 'by', 'from', 'at', 'that',
    'this', 'are', 'as', 'be', 'or', 'will', 'have', 'had', 'were',
    'been', 'not', 'but', 'which', 'their', 'than', 'also', 'said',
    'would', 'about', 'more', 'during', 'per', 'eur', 'million',
    'company', 'year', 'period'
}

print("TOP 10 PALABRAS POR SENTIMIENTO")
print("="*50)
for sentiment in ["positive", "negative", "neutral"]:
    texts = " ".join(df[df["sentiment"] == sentiment]["sentence"].values)
    words = re.findall(r'[a-z]+', texts.lower())
    words = [w for w in words if w not in stopwords and len(w) > 2]
    top = Counter(words).most_common(10)

    print(f"\n  {sentiment.upper()}:")
    for word, count in top:
        bar = "█" * (count // 10)
        print(f"    {word:20s} {count:4d} {bar}")
        
        
# Si las frases negativas fueran mucho más largas,
# la longitud misma sería una feature útil.
df["word_count"] = df["sentence"].apply(lambda x: len(str(x).split()))

print(f"\nLONGITUD DE FRASES (palabras):")
for sentiment in ["positive", "negative", "neutral"]:
    subset = df[df["sentiment"] == sentiment]["word_count"]
    print(f"  {sentiment:10s} → media: {subset.mean():.1f} | "
          f"min: {subset.min()} | max: {subset.max()}")

TOP 10 PALABRAS POR SENTIMIENTO

  POSITIVE:
    finnish               202 ████████████████████
    net                   196 ███████████████████
    sales                 192 ███████████████████
    profit                192 ███████████████████
    mln                   128 ████████████
    operating             122 ████████████
    quarter               114 ███████████
    oyj                    97 █████████
    group                  97 █████████
    rose                   94 █████████

  NEGATIVE:
    profit                156 ███████████████
    net                   104 ██████████
    finnish               102 ██████████
    sales                  98 █████████
    operating              97 █████████
    quarter                89 ████████
    down                   83 ████████
    loss                   69 ██████
    compared               68 ██████
    decreased              68 ██████

  NEUTRAL:
    finland               228 ██████████████████████
    finnish               220 █

TF-IDF (Term Frequency x Inverse Document Frequency)

TF (Term Frequency): ¿Cuántas veces aparece esta palabra en ESTA frase?
  → "profit" aparece 2 veces → TF alto

IDF (Inverse Document Frequency): ¿Es rara esta palabra en el corpus?
  → "the" aparece en TODAS las frases → IDF bajo (no aporta info)
  → "restructuring" aparece en pocas → IDF alto (es discriminativa)

TF-IDF = TF × IDF
  → Palabras frecuentes en UNA frase pero raras en el corpus
    obtienen puntuación alta → son las más informativas.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Limpiar texto
def clean_text(text):
    text = str(text).lower()                    # minúsculas
    text = re.sub(r'\d+\.?\d*', ' NUM ', text)  # números → token genérico
    text = re.sub(r'[^\w\s]', ' ', text)        # quitar puntuación
    text = re.sub(r'\s+', ' ', text).strip()    # espacios extra
    return text

df["clean_text"] = df["sentence"].apply(clean_text)

# Antes y después de la limpieza de texto

print("LIMPIEZA DE TEXTO - Antes vs Después:")
for i in range(3):
    print(f"  ANTES:   {df['sentence'].iloc[i][:80]}")
    print(f"  DESPUÉS: {df['clean_text'].iloc[i][:80]}")
    print()


# Vectorizador TF-IDF

tfidf = TfidfVectorizer(
    max_features=5000,   # solo las 5000 palabras más frecuentes
    ngram_range=(1, 2),  # unigrams Y bigrams ("net profit" como una feature)
    min_df=2,            # ignorar palabras que aparecen en menos de 2 docs
    max_df=0.95,         # ignorar palabras que aparecen en más del 95% de docs
    stop_words='english' # quitar stopwords automáticamente
)
